In [12]:
import sys
import os
import torch
import numpy as np
import pandas as pd
from tqdm import tqdm

In [13]:
df = pd.read_csv('/home/jingjie/AutoTorch/src/eval/idfraud/annotation/dataframes/processed_recapture_tamper_feb_processed.csv')

df.shape

(5596, 48)

In [14]:
def calculate_apcer_bpcer(df, pred_col='parallel_label', gt_col='annot_isfraud'):
    """
    Calculate APCER and BPCER for binary classification.
    
    APCER = FN / (TP + FN) = attacks misclassified as genuine / total attacks
    BPCER = FP / (TN + FP) = genuine misclassified as attacks / total genuine
    
    Args:
        df: DataFrame with predictions and ground truth
        pred_col: Column name for model predictions (0=genuine, 1=attack)
        gt_col: Column name for ground truth labels
    """
    # Filter out unknown labels
    data = df[df[gt_col] != -1]
    
    # Ground truth
    attacks = data[data[gt_col] == 1]  # actual attacks (fraud)
    genuine = data[data[gt_col] == 0]  # actual genuine (bona fide)
    
    # APCER: attacks incorrectly classified as genuine
    fn = (attacks[pred_col] == 0).sum()  # false negatives
    apcer = fn / len(attacks) if len(attacks) > 0 else 0
    
    # BPCER: genuine incorrectly classified as attacks  
    fp = (genuine[pred_col] == 1).sum()  # false positives
    bpcer = fp / len(genuine) if len(genuine) > 0 else 0
    
    return {
        'APCER': apcer,
        'BPCER': bpcer,
        'Total Attacks': len(attacks),
        'Total Genuine': len(genuine),
        'FN (missed attacks)': fn,
        'FP (false alarms)': fp
    }

In [ ]:
for model in ['exp2point5_ori_label','exp2point5_crop_label', 'exp2point5_parallel_label', 'exp5_parallel_label','exp3_parallel_label', 'exp4_parallel_label', 'exp7_parallel_label']:
    metrics = calculate_apcer_bpcer(df, pred_col=model, gt_col='annot_is_idfraud')
    print(f"\n{model}:")
    print(f"  APCER: {metrics['APCER']:.4f} ({metrics['FN (missed attacks)']}/{metrics['Total Attacks']})")
    print(f"  BPCER: {metrics['BPCER']:.4f} ({metrics['FP (false alarms)']}/{metrics['Total Genuine']})")


exp2point5_ori_label:
  APCER: 0.0927 (46/496)
  BPCER: 0.0765 (389/5084)

exp2point5_crop_label:
  APCER: 0.0887 (44/496)
  BPCER: 0.2364 (1202/5084)

exp2point5_parallel_label:
  APCER: 0.0786 (39/496)
  BPCER: 0.1408 (716/5084)

exp5_parallel_label:
  APCER: 0.1048 (52/496)
  BPCER: 0.1174 (597/5084)

exp3_parallel_label:
  APCER: 0.0948 (47/496)
  BPCER: 0.0913 (464/5084)

exp4_parallel_label:
  APCER: 0.0524 (26/496)
  BPCER: 0.2331 (1185/5084)


In [16]:
df_exclude_quality = df.loc[df['annot_is_Quality']==0]
df_exclude_quality.shape

(3330, 48)

In [17]:
df['annot_is_Quality'].value_counts()

annot_is_Quality
False    3330
True     2266
Name: count, dtype: int64

In [20]:
for model in ['exp2point5_ori_label','exp2point5_crop_label', 'exp2point5_parallel_label','exp5_parallel_label','exp3_parallel_label','exp4_parallel_label']:
    metrics = calculate_apcer_bpcer(df_exclude_quality, pred_col=model, gt_col='annot_is_idfraud')
    print(f"\n{model}:")
    print(f"  APCER: {metrics['APCER']:.4f} ({metrics['FN (missed attacks)']}/{metrics['Total Attacks']})")
    print(f"  BPCER: {metrics['BPCER']:.4f} ({metrics['FP (false alarms)']}/{metrics['Total Genuine']})")


exp2point5_ori_label:
  APCER: 0.0927 (46/496)
  BPCER: 0.0443 (125/2823)

exp2point5_crop_label:
  APCER: 0.0887 (44/496)
  BPCER: 0.1183 (334/2823)

exp2point5_parallel_label:
  APCER: 0.0786 (39/496)
  BPCER: 0.0670 (189/2823)

exp5_parallel_label:
  APCER: 0.1048 (52/496)
  BPCER: 0.0457 (129/2823)

exp3_parallel_label:
  APCER: 0.0948 (47/496)
  BPCER: 0.0446 (126/2823)

exp4_parallel_label:
  APCER: 0.0524 (26/496)
  BPCER: 0.1151 (325/2823)


In [22]:
def threshold_search(df, ori_col, crop_col, gt_col='annot_is_idfraud', thresholds=None):
    """
    Search for optimal threshold on averaged ori+crop predictions.
    Returns DataFrame with APCER, BPCER, ACER for each threshold.
    """
    if thresholds is None:
        thresholds = np.arange(0.1, 0.9, 0.05)
    
    # Average predictions
    avg_pred = (df[ori_col] + df[crop_col]) / 2
    
    results = []
    for thresh in thresholds:
        preds = (avg_pred > thresh).astype(int)
        
        # Filter valid labels
        valid = df[gt_col] != -1
        attacks = df[valid & (df[gt_col] == 1)]
        genuine = df[valid & (df[gt_col] == 0)]
        
        # APCER: attacks missed (pred=0 when gt=1)
        fn = (preds[attacks.index] == 0).sum()
        apcer = fn / len(attacks) if len(attacks) > 0 else 0
        
        # BPCER: genuine flagged (pred=1 when gt=0)
        fp = (preds[genuine.index] == 1).sum()
        bpcer = fp / len(genuine) if len(genuine) > 0 else 0
        
        results.append({
            'threshold': thresh,
            'APCER': apcer,
            'BPCER': bpcer,
            'ACER': (apcer + bpcer) / 2
        })
    
    return pd.DataFrame(results)


# Run threshold search for each experiment
experiments = [
    ('exp2point5', 'exp2point5_pred_ori', 'exp2point5_pred_crop'),
    ('exp5', 'exp5_pred_ori', 'exp5_pred_crop'),
    ('exp3', 'exp3_pred_ori', 'exp3_pred_crop'),
]

print("=" * 60)
print("THRESHOLD SEARCH (on all data)")
print("=" * 60)

for exp_name, ori_col, crop_col in experiments:
    if ori_col not in df.columns or crop_col not in df.columns:
        print(f"\n{exp_name}: Missing prediction columns, skipping")
        continue
    
    results = threshold_search(df, ori_col, crop_col)
    best = results.loc[results['ACER'].idxmin()]
    
    print(f"\n{exp_name}:")
    display(results)
    print(f"Best threshold: {best['threshold']:.2f} (APCER: {best['APCER']:.4f}, BPCER: {best['BPCER']:.4f}, ACER: {best['ACER']:.4f})")

print("\n" + "=" * 60)
print("THRESHOLD SEARCH (excluding quality issues)")
print("=" * 60)

for exp_name, ori_col, crop_col in experiments:
    if ori_col not in df_exclude_quality.columns or crop_col not in df_exclude_quality.columns:
        print(f"\n{exp_name}: Missing prediction columns, skipping")
        continue
    
    results = threshold_search(df_exclude_quality, ori_col, crop_col)
    best = results.loc[results['ACER'].idxmin()]
    
    print(f"\n{exp_name}:")
    display(results)
    print(f"Best threshold: {best['threshold']:.2f} (APCER: {best['APCER']:.4f}, BPCER: {best['BPCER']:.4f}, ACER: {best['ACER']:.4f})")

THRESHOLD SEARCH (on all data)

exp2point5:


,threshold,APCER,BPCER,ACER
0,0.10,0.040323,0.302124,0.171223
1,0.15,0.048387,0.284028,0.166208
2,0.20,0.050403,0.272227,0.161315
3,0.25,0.054435,0.260228,0.157332
4,0.30,0.056452,0.249607,0.153029
5,0.35,0.056452,0.235051,0.145751
6,0.40,0.058468,0.220692,0.139580
7,0.45,0.060484,0.196105,0.128295
8,0.50,0.078629,0.140834,0.109732
9,0.55,0.102823,0.093430,0.098126


Best threshold: 0.75 (APCER: 0.1290, BPCER: 0.0545, ACER: 0.0918)

exp5:


,threshold,APCER,BPCER,ACER
0,0.10,0.042339,0.250787,0.146563
1,0.15,0.042339,0.240755,0.141547
2,0.20,0.042339,0.233871,0.138105
3,0.25,0.042339,0.228363,0.135351
4,0.30,0.044355,0.220299,0.132327
5,0.35,0.048387,0.211054,0.129721
6,0.40,0.060484,0.202203,0.131343
7,0.45,0.072581,0.184304,0.128442
8,0.50,0.104839,0.117427,0.111133
9,0.55,0.153226,0.081432,0.117329


Best threshold: 0.50 (APCER: 0.1048, BPCER: 0.1174, ACER: 0.1111)

exp3:


...

Best threshold: 0.50 (APCER: 0.0786, BPCER: 0.0670, ACER: 0.0728)

exp5:


,threshold,APCER,BPCER,ACER
0,0.10,0.042339,0.125399,0.083869
1,0.15,0.042339,0.117605,0.079972
2,0.20,0.042339,0.114063,0.078201
3,0.25,0.042339,0.109104,0.075721
4,0.30,0.044355,0.104853,0.074604
5,0.35,0.048387,0.100956,0.074672
6,0.40,0.060484,0.094226,0.077355
7,0.45,0.072581,0.083953,0.078267
8,0.50,0.104839,0.045696,0.075267
9,0.55,0.153226,0.030464,0.091845


Best threshold: 0.30 (APCER: 0.0444, BPCER: 0.1049, ACER: 0.0746)

exp3:


,threshold,APCER,BPCER,ACER
0,0.10,0.042339,0.176054,0.109196
1,0.15,0.042339,0.166135,0.104237
2,0.20,0.042339,0.154446,0.098392
3,0.25,0.042339,0.145944,0.094141
4,0.30,0.046371,0.136734,0.091552
5,0.35,0.048387,0.122210,0.085299
6,0.40,0.050403,0.108395,0.079399
7,0.45,0.056452,0.086433,0.071442
8,0.50,0.094758,0.044633,0.069696
9,0.55,0.116935,0.032944,0.074940


Best threshold: 0.50 (APCER: 0.0948, BPCER: 0.0446, ACER: 0.0697)


In [ ]:
import ast 
# convert string representation of list to actual Python list
df["tag"] = df["tag"].apply(ast.literal_eval)
df_exclude_quality["tag"] = df_exclude_quality["tag"].apply(ast.literal_eval)

print(df["tag"].iloc[0])
print(type(df["tag"].iloc[0]))

['Genuine']
<class 'list'>


/tmp/ipykernel_135788/3749275958.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_exclude_quality["tag"] = df_exclude_quality["tag"].apply(ast.literal_eval)


In [ ]:

def breakdown_by_tag(df, models=['exp2point5_ori_label','exp2point5_crop_label', 'exp2point5_parallel_label','exp5_parallel_label']):

    fraud_tags = [
            'Printed - Grayscale',
            'Printed - Color',
            'Cutout Printed - Grayscale',
            'Cutout Printed - Color',
            'Replay - Mobile',
            'Replay - Tablet',
            'Replay - Monitor',
            'Fullpage Printed - Color',
        ]
    
    rows = []
    for tag in fraud_tags + ['Genuine']:
        tag_df = df[df['tag'].apply(lambda x: tag in x)]
        if len(tag_df) == 0:
            continue
        row = {'tag': tag, 'count': len(tag_df)}
        for model in models:
            # For fraud: miss rate (pred=0), For genuine: false alarm rate (pred=1)
            # sum the preds==0 (model classifies as genuine when the tag is fraud) ; 
            error = (tag_df[model] == 0).sum() if tag != 'Genuine' else (tag_df[model] == 1).sum()
            row[model] = f"{error/len(tag_df):.2%} ({error}/{len(tag_df)})"
        rows.append(row)
    
    return pd.DataFrame(rows).set_index('tag')

# miss rate
print("=== Breakdown statistics: APCER for fraud and BPCER for genuine ===")
print("Exclude Quality images:")
display(breakdown_by_tag(df).reset_index())
print("All images:")
display(breakdown_by_tag(df_exclude_quality).reset_index())


=== Breakdown statistics: APCER for fraud and BPCER for genuine ===
Exclude Quality images:


,tag,count,exp2point5_ori_label,exp2point5_crop_label,exp2point5_parallel_label,exp5_parallel_label
0,Printed - Grayscale,17,5.88% (1/17),0.00% (0/17),0.00% (0/17),0.00% (0/17)
1,Printed - Color,25,52.00% (13/25),64.00% (16/25),52.00% (13/25),60.00% (15/25)
2,Cutout Printed - Grayscale,4,0.00% (0/4),0.00% (0/4),0.00% (0/4),50.00% (2/4)
3,Cutout Printed - Color,34,38.24% (13/34),14.71% (5/34),17.65% (6/34),26.47% (9/34)
4,Replay - Mobile,345,4.93% (17/345),6.09% (21/345),5.22% (18/345),6.67% (23/345)
5,Replay - Tablet,6,16.67% (1/6),16.67% (1/6),16.67% (1/6),16.67% (1/6)
6,Replay - Monitor,65,1.54% (1/65),1.54% (1/65),1.54% (1/65),3.08% (2/65)
7,Genuine,5086,7.69% (391/5086),23.67% (1204/5086),14.12% (718/5086),11.78% (599/5086)


All images:


,tag,count,exp2point5_ori_label,exp2point5_crop_label,exp2point5_parallel_label,exp5_parallel_label
0,Printed - Grayscale,17,5.88% (1/17),0.00% (0/17),0.00% (0/17),0.00% (0/17)
1,Printed - Color,25,52.00% (13/25),64.00% (16/25),52.00% (13/25),60.00% (15/25)
2,Cutout Printed - Grayscale,4,0.00% (0/4),0.00% (0/4),0.00% (0/4),50.00% (2/4)
3,Cutout Printed - Color,34,38.24% (13/34),14.71% (5/34),17.65% (6/34),26.47% (9/34)
4,Replay - Mobile,345,4.93% (17/345),6.09% (21/345),5.22% (18/345),6.67% (23/345)
5,Replay - Tablet,6,16.67% (1/6),16.67% (1/6),16.67% (1/6),16.67% (1/6)
6,Replay - Monitor,65,1.54% (1/65),1.54% (1/65),1.54% (1/65),3.08% (2/65)
7,Genuine,2825,4.50% (127/2825),11.89% (336/2825),6.76% (191/2825),4.64% (131/2825)
